In [ ]:
import pandas as pd
import time
import re
import os
import sempy.fabric as fabric 
from fabric.dataagent.client import FabricOpenAI
from fabric.dataagent.client import delete_data_agent
from datetime import datetime
from pyspark.sql.functions import lit

In [ ]:
%run 2. Parameters

In [ ]:
%run 3. M Code Highlighter

In [ ]:
df = spark.sql("SELECT * FROM SMD_LH.list_columns LIMIT 1000")
display(df)

In [ ]:
df = spark.sql("SELECT * FROM SMD_LH.r_partitions LIMIT 1000")
display(df)

In [ ]:
df = spark.sql("SELECT * FROM SMD_LH.r_relationships LIMIT 1000")
display(df)

In [ ]:
list_columns_df = spark.sql("SELECT * FROM SMD_LH.list_columns LIMIT 1000")
display(list_columns_df)

list_r_partitions_df = spark.sql("SELECT * FROM SMD_LH.r_partitions LIMIT 1000")
display(list_r_partitions_df)


list_r_relationships_doubled_df = spark.sql("SELECT * FROM SMD_LH.r_relationships LIMIT 1000")
display(list_r_relationships_doubled_df)

# Perform right join using PySpark
combined_df = list_columns_df.join(
    list_r_partitions_df, 
    list_columns_df.TABLE_NAME == list_r_partitions_df.T_NAME,
    how='right'
).select(
    *[list_columns_df[col] for col in list_columns_df.columns if col != 'DATABASE_NAME'],  # All columns except DATABASE_NAME
    list_r_partitions_df.T_NAME,
    list_r_partitions_df.P_QUERYDEFINITION,
    list_r_partitions_df.DATABASE_NAME
)

display(combined_df)

In [ ]:
from pyspark.sql import functions as F

# Join relationships where TABLE_NAME matches either FROM_TABLE or TO_TABLE
combined_with_relationships_df = combined_df.join(
    list_r_relationships_doubled_df,
    (combined_df.TABLE_NAME == list_r_relationships_doubled_df.FROM_TABLE) |
    (combined_df.TABLE_NAME == list_r_relationships_doubled_df.TO_TABLE),
    how='left'
).select(
    *[combined_df[col] for col in combined_df.columns],
    list_r_relationships_doubled_df.FROM_TABLE,
    list_r_relationships_doubled_df.FROM_COLUMN,
    list_r_relationships_doubled_df.TO_TABLE,
    list_r_relationships_doubled_df.TO_COLUMN,
    list_r_relationships_doubled_df.MULTIPLICITY,
    list_r_relationships_doubled_df.ACTIVE
)

display(combined_with_relationships_df)

In [ ]:
import pandas as pd

# Convert PySpark DataFrame to Pandas first
combined_df = combined_with_relationships_df.toPandas()

# Check how many nulls exist
print("Null count:", combined_df['P_QUERYDEFINITION'].isnull().sum())

# Apply functions with None handling
combined_df['SSRS_HTML'] = combined_df['P_QUERYDEFINITION'].apply(
    lambda x: mcode_to_ssrs_html(x) if pd.notna(x) else ""
)
combined_df['MCODE_DISPLAY'] = combined_df['P_QUERYDEFINITION'].apply(
    lambda x: display_mcode(x) if pd.notna(x) else ""
)

display(combined_df)

In [ ]:
import pandas as pd

# Convert PySpark DataFrame to Pandas first
combined_df = combined_with_relationships_df.toPandas()

# Now apply your functions on the Pandas DataFrame
combined_df['SSRS_HTML'] = combined_df['P_QUERYDEFINITION'].apply(mcode_to_ssrs_html)
combined_df['MCODE_DISPLAY'] = combined_df['P_QUERYDEFINITION'].apply(display_mcode)

display(combined_df)

In [ ]:
import pandas as pd
from datetime import datetime
from pyspark.sql.types import StringType, StructField, StructType

# combined_df is already a Pandas DataFrame
pandas_df = combined_df.copy()

# Convert all column names to uppercase
pandas_df.columns = pandas_df.columns.str.upper()

# Add timestamp column
pandas_df["DATABASE_TIMESTAMP"] = f"{semantic_model_name} | {datetime.now():%d.%m.%Y %H:%M:%S}"

# Determine version number based on existing data
table_name = "column_metadata"

if spark.catalog.tableExists(table_name):
    existing_df = spark.read.table(table_name)
    
    max_version_row = existing_df.filter(
        existing_df.DATABASE_VERSION.startswith(semantic_model_name)
    ).selectExpr("max(cast(regexp_extract(DATABASE_VERSION, 'V(\\\\d+)', 1) as int)) as max_ver").collect()
    
    max_version = max_version_row[0]['max_ver'] if max_version_row else None
    next_version = (max_version + 1) if max_version is not None else 1
    write_mode = "append"
else:
    next_version = 1
    write_mode = "overwrite"

# Add version column
pandas_df["DATABASE_VERSION"] = f"{semantic_model_name} | V{next_version}"

# ── Fix void/null-only columns before converting to Spark ──────────────────
# Cast any object or void columns to string to avoid Spark schema inference errors
for col in pandas_df.columns:
    if pandas_df[col].dtype == object or str(pandas_df[col].dtype) == 'void':
        pandas_df[col] = pandas_df[col].astype(str).replace('None', None).replace('nan', None)

# Build an explicit schema so Spark doesn't have to guess
schema = StructType([
    StructField(col, StringType(), True) for col in pandas_df.columns
])

# Convert back to Spark only for the write step — using explicit schema
spark_df = spark.createDataFrame(pandas_df, schema=schema)
spark_df.write.option("mergeSchema", "true").saveAsTable(name=table_name, mode=write_mode)

print(f"Successfully wrote to table '{table_name}' with version V{next_version}")
display(pandas_df)

In [ ]:
df = spark.sql("SELECT * FROM SMD_LH.column_metadata LIMIT 1000")
display(df)